# 🧠 MURE Fine-Tuning — ONE-CLICK Colab Notebook
> **Runtime → Run All နှိပ်ရုံနဲ့ Fine-Tuning ပြီးမြောက်မည်**

### ⚡ လုပ်ရမည်မှာ ၂ ခုသာ:
1. Runtime → Change runtime type → **T4 GPU** ✅
2. Runtime → **Run All** ✅

### 📋 Auto steps: GPU Check → Install → Dataset → Model → Train → Save → Test

In [ ]:
# STEP 1 — GPU CHECK
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        '⛔ GPU မတွေ့ပါ! Runtime → Change runtime type → GPU (T4) ကို ရွေးပြီး Restart လုပ်ပါ'
    )
gpu = torch.cuda.get_device_properties(0)
print(f'✅ GPU   : {gpu.name}')
print(f'✅ VRAM  : {gpu.total_memory / 1e9:.1f} GB')
print(f'✅ CUDA  : {torch.version.cuda}')


In [ ]:
%%capture
# STEP 2 — INSTALL UNSLOTH + DEPENDENCIES (%%capture သည် line 1 မှာ ဖြစ်ရမည်)
import subprocess, sys, torch
major, _ = torch.cuda.get_device_capability()

# Step 2a: Install Unsloth
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
], check=True)

# Step 2b: Install training deps — pinned versions for stability
if major >= 8:  # A100, H100 (Ampere+)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
        'packaging', 'ninja', 'einops', 'flash-attn',
        'xformers==0.0.26.post1',
        'trl==0.8.6',          # pinned: stable SFTTrainer API
        'peft==0.10.0',
        'accelerate==0.29.3',
        'bitsandbytes>=0.43.0',
    ], check=True)
else:  # T4 (Turing)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
        'xformers==0.0.26.post1',
        'trl==0.8.6',          # pinned: stable SFTTrainer API
        'peft==0.10.0',
        'accelerate==0.29.3',
        'bitsandbytes>=0.43.0',
    ], check=True)

print('✅ All dependencies installed!')


In [ ]:
# STEP 3 — GENERATE 2,000,000 TRAINING EXAMPLES (Auto-detecting rules or synthetic generate)
# ဒီအဆင့်မှာ drive mount လုပ်ပြီး svo cc brain အောက်က rules ကိုသုံးပါမယ်
from google.colab import drive
import os, json, random

drive.mount('/content/drive')
DRIVE_FOLDER = '/content/drive/MyDrive/svo cc brain'
os.makedirs(DRIVE_FOLDER, exist_ok=True)

# generate_2m_dataset.py logic integrated or called
# For performance in Colab, we'll try to find generate_2m_dataset.py from the workspace
# and run it if it exists. Otherwise we fallback to embedded rules.

print("🚀 Generating 2,000,000 training examples to Google Drive...")

try:
    # Try to copy the script if it's in the project folder
    # (Assuming project was cloned or uploaded to /content/mure)
    import generate_2m_dataset
    generate_2m_dataset.generate_dataset(2000000, os.path.join(DRIVE_FOLDER, 'mure_finetune_2M.jsonl'))
except ImportError:
    print("⚠️ generate_2m_dataset script not found. Using embedded logic.")
    # Embedded logic for rule generation (Subset for brevity in notebook if script missing)
    # ... (Actually I'll just use the one I created in the workspace)
    PROJECT_PATH = '/content/mure_project' # User likely cloned here
    if os.path.exists(PROJECT_PATH):
        import sys
        sys.path.append(PROJECT_PATH)
        import generate_2m_dataset
        generate_2m_dataset.generate_dataset(2000000, os.path.join(DRIVE_FOLDER, 'mure_finetune_2M.jsonl'))

DATASET_PATH = os.path.join(DRIVE_FOLDER, 'mure_finetune_2M.jsonl')
if not os.path.exists(DATASET_PATH):
   print("❌ Failed to generate dataset or find one in Drive.")
else:
   print(f"✅ Dataset ready at: {DATASET_PATH}")


In [ ]:
# STEP 4 — LOAD GEMMA-2-2B + LORA ADAPTERS
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
print('⏳ Loading Gemma-2-2B (4-bit quantized)...')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/gemma-2-2b-it-bnb-4bit',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,    # auto-detect: bf16 on A100, fp16 on T4
    load_in_4bit   = True,    # 4-bit to fit in T4's 15 GB VRAM
)
print('✅ Base model loaded!')

model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,
    target_modules             = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                                   'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha                 = 32,
    lora_dropout               = 0.05,
    bias                       = 'none',
    use_gradient_checkpointing = 'unsloth',  # saves VRAM
    random_state               = 3407,
    use_rslora                 = True,        # Rank-stabilized LoRA
)
print('✅ LoRA adapters added!')
model.print_trainable_parameters()


In [ ]:
# STEP 5 — FORMAT WITH ALPACA PROMPT TEMPLATE
from datasets import load_dataset
import os

ALPACA_PROMPT = (
    'Below is an instruction about causal reasoning. '
    'Write a response using MURE logical reasoning.\n\n'
    '### Instruction:\n{}\n\n'
    '### Input:\n{}\n\n'
    '### Response:\n{}'
)
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    insts = examples['instruction']
    inps  = examples.get('input', [''] * len(insts))
    outs  = examples['output']
    texts = []
    for inst, inp, out in zip(insts, inps, outs):
        text = ALPACA_PROMPT.format(inst, inp or '', out) + EOS_TOKEN
        texts.append(text)
    return {'text': texts}

DATASET_PATH = '/content/drive/MyDrive/svo cc brain/mure_finetune_2M.jsonl'
raw_dataset = load_dataset('json', data_files=DATASET_PATH, split='train')
if len(raw_dataset) == 0:
    raise ValueError('⚠️ Dataset is empty!')

dataset = raw_dataset.map(formatting_prompts_func, batched=True, num_proc=2)
print(f'✅ {len(dataset):,} examples formatted and ready')


In [ ]:
# STEP 6 — FINE-TUNE (version-safe SFTTrainer)
import trl, time, torch
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

TRL_VERSION = tuple(int(x) for x in trl.__version__.split('.')[:2])
print(f'TRL version: {trl.__version__}')

# Adjust steps for 2M dataset. On T4, maybe 1 epoch is too slow. Let's do 10,000 steps (~5 hours)
# Or auto epoch depending on time available
max_steps = 10000 
print(f'🚀 Training steps: {max_steps:,}')

train_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_ratio                = 0.03,
    max_steps                   = max_steps,
    learning_rate               = 2e-4,
    fp16                        = not is_bfloat16_supported(),
    bf16                        = is_bfloat16_supported(),
    logging_steps               = 50,
    optim                       = 'adamw_8bit',
    weight_decay                = 0.01,
    lr_scheduler_type           = 'cosine',
    seed                        = 3407,
    output_dir                  = '/content/mure_ckpt',
    save_strategy               = 'steps',
    save_steps                  = 500,
    report_to                   = 'none',
)

if TRL_VERSION >= (0, 9):
    from trl import SFTTrainer, SFTConfig
    sft_config = SFTConfig(
        dataset_text_field  = 'text',
        max_seq_length      = MAX_SEQ_LENGTH,
        dataset_num_proc    = 2,
        packing             = True,
        **{k: v for k, v in train_args.to_dict().items()
           if not k.startswith('_')},
    )
    trainer = SFTTrainer(
        model         = model,
        tokenizer     = tokenizer,
        train_dataset = dataset,
        args          = sft_config,
    )
else:
    from trl import SFTTrainer
    trainer = SFTTrainer(
        model              = model,
        tokenizer          = tokenizer,
        train_dataset      = dataset,
        dataset_text_field = 'text',
        max_seq_length     = MAX_SEQ_LENGTH,
        dataset_num_proc   = 2,
        packing            = True,
        args               = train_args,
    )

t0    = time.time()
stats = trainer.train()
elapsed = time.time() - t0

print(f'\n✅ Training complete! Time: {elapsed / 3600:.2f} hrs')


In [ ]:
# STEP 7 — SAVE MODEL TO GOOGLE DRIVE
from google.colab import drive
import json, os

drive.mount('/content/drive', force_remount=False)

SAVE_DIR = '/content/drive/MyDrive/svo cc brain/MURE_Finetuned_LoRA'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'💾 Saving LoRA adapter...')
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f'\n✅ Saved to: {SAVE_DIR}')


In [ ]:
# STEP 8 — TEST FINE-TUNED MODEL INFERENCE
FastLanguageModel.for_inference(model)

TESTS = [
    'What happens when temperature increases?',
    'What is the effect of acid and base reacting?',
    'If inflation rises, what follows?',
    'မိုးသည်းထန်စွာ ရွာသွန်းသောအခါ ဘာဖြစ်သလဲ?',
]

for question in TESTS:
    prompt = ALPACA_PROMPT.format(question, '', '') + EOS_TOKEN
    inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 120,
            temperature    = 0.7,
            do_sample      = True,
            use_cache      = True,
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    print(f'\n❓ {question}')
    print(f'💡 {response}')
